# A3.3.3: Updating data with SQL

*IB Computer Science 2027, topic A3 Databases, A3.3 Database programming.* Core for SL and HL. Command term: **construct and explain**. Builds on A3.3.1.

*Three statements change what is stored. A missing WHERE is the most expensive mistake in SQL, and an index makes every update do extra work.*

**How to use this notebook**
- Click a code cell and press **Shift+Enter** to run it. Run the cells **in order, from the top**.
- A cell that starts with `%%sql` holds plain SQL. A cell with `<<` after `%%sql` saves its result so the check cell can look at it. A cell that creates or changes things shows an empty box. That is normal.
- A **TODO** cell is yours. Each starts with a placeholder line `SELECT 'write your SQL here' AS todo;`. Delete that line when you have written your own SQL.
- The **check** cells print ✓ when you are right and ✗ with the difference when you are not.
- The database lives in memory. If you restart the kernel, run the cells again from the top.

Notes adapted from the ExamEdge page https://examedge.app/learn/ib/a3-3-3

In [ ]:
# Helper functions for the check cells. Run this once. You do not need to read it.
def _rows(var):
    if var not in globals():
        return None
    return [tuple(r) for r in globals()[var]]

def _norm(rows, places=2):
    return [tuple(round(v, places) if isinstance(v, float) else v for v in r) for r in rows]

def _compare(name, got, want, ordered):
    g, w = _norm(got), _norm(want)
    ok = (g == w) if ordered else (sorted(g, key=repr) == sorted(w, key=repr))
    if ok:
        print("✓", name, "is correct.")
    else:
        print("✗", name, "is not right yet.")
        print("   You returned", len(g), "row(s):", g[:6])
        print("   Expected    ", len(w), "row(s):", w[:6])

def check_rows(name, var, want, ordered=False):
    # Compares the rows saved by a "%%sql var <<" cell with the expected rows.
    got = _rows(var)
    if got is None:
        print("✗", name, ": there is no result yet. Run your SQL cell first.")
        return
    _compare(name, got, want, ordered)

def check_db(name, sql, want, ordered=False):
    # Runs a question against the practice database and compares the rows.
    try:
        got = [tuple(r) for r in db.execute(sql).fetchall()]
    except Exception as e:
        print("✗", name, ": my check could not run:", e)
        return
    _compare(name, got, want, ordered)

def check_choice(choice, correct, why_right, why_wrong):
    c = str(choice).strip().upper()
    if c not in "ABCD" or c == "":
        print("Set choice to A, B, C or D, then run this cell again.")
    elif c == correct:
        print("✓ Correct.", why_right)
    else:
        print("✗ Not quite.", why_wrong)

def check_dict(name, got, want):
    bad = [k for k in want if str(got.get(k, "")).strip().upper() != want[k]]
    if not bad:
        print("✓", name, "all correct.")
    else:
        print("✗", name, ":", len(bad), "of", len(want), "not right yet:")
        for k in bad:
            print("    ", k)

In [ ]:
# Install the SQL add-on into THIS kernel if it is missing (one time only). (Given.)
import importlib.util, subprocess, sys
if importlib.util.find_spec("sql") is None:
    print("Installing the SQL add-on (one time, about a minute)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupysql"])
    importlib.invalidate_caches()

import sqlite3
db = sqlite3.connect(":memory:")

def reset():
    # Puts the SEEDLING table back to how it started (with its index).
    db.executescript('''
    DROP TABLE IF EXISTS Seedling;
    CREATE TABLE Seedling (SeedlingID TEXT PRIMARY KEY, SpeciesName TEXT, TrayCode TEXT, StockCount INTEGER, Price REAL);
    INSERT INTO Seedling VALUES ('S1', 'Blue Fescue', 'T-04', 120, 3.50),
                                ('S2', 'Coral Bells', 'T-11', 45, 5.20),
                                ('S3', 'Silver Thyme', 'T-04', 80, 2.80),
                                ('S4', 'Wax Begonia', 'T-07', 0, 4.10);
    CREATE INDEX idx_species ON Seedling (SpeciesName);
    ''')

reset()

%load_ext sql
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = False
%sql db

## 1. Update queries

An **update query** is any DML statement that changes the contents of a table rather than just reading them. There are three: `INSERT INTO` adds records, `UPDATE ... SET` modifies existing records, and `DELETE` removes records. None of them touch the structure. Only the rows change.

A small nursery keeps one table of what is growing. Here it is.

In [ ]:
%%sql
SELECT * FROM Seedling;

**INSERT** adds a new record. Name the columns, then give one value per column in the same order. The primary key must be present and unique.

In [ ]:
%%sql
INSERT INTO Seedling (SeedlingID, SpeciesName, TrayCode, StockCount, Price)
VALUES ('S5', 'Sea Holly', 'T-11', 60, 6.00);
SELECT * FROM Seedling;

**UPDATE ... SET ... WHERE** changes values in the rows that match. **DELETE FROM ... WHERE** removes the rows that match.

In [ ]:
%%sql
UPDATE Seedling SET Price = 3.00 WHERE SeedlingID = 'S3';
SELECT * FROM Seedling;

In [ ]:
%%sql
DELETE FROM Seedling WHERE StockCount = 0;
SELECT * FROM Seedling;

## 2. WHERE is what limits the damage

`UPDATE` and `DELETE` apply to every row that matches the condition. **If the condition is left out, every row matches.** Neither statement warns you first.

**Predict:** what will this do to the Price column? Then run it.

In [ ]:
%%sql
UPDATE Seedling SET Price = 3.00;
SELECT * FROM Seedling;

Every price became 3.00. Now put the table back.

In [ ]:
reset()
print("Table reset.")

Good habit: run the `SELECT` with the intended `WHERE` first, to see exactly which rows will be affected, then change `SELECT *` into `UPDATE ... SET` or `DELETE`.

In [ ]:
%%sql
SELECT * FROM Seedling WHERE StockCount = 0;

## 3. Why updating an indexed column costs more

An **index** is a separate, sorted structure the database keeps next to a table so a search on one column does not scan every row. `SpeciesName` is indexed here. SQLite can tell us when it uses the index.

In [ ]:
%%sql
EXPLAIN QUERY PLAN SELECT * FROM Seedling WHERE SpeciesName = 'Coral Bells';

The plan says `SEARCH ... USING INDEX idx_species`. Now compare two updates. Updating `Price` touches the row only. Updating `SpeciesName` changes the row **and** the index: the old index entry is removed and a new one inserted in the right sorted position.

- Many updates on an indexed column leave the index **fragmented**: still correct, but spread over more space than it needs, so reads get slower.
- A database administrator manages this by **batching** updates, **rebuilding or reorganizing** the index regularly (SQLite: `REINDEX`), and using **partial indexes** that cover only the rows actually searched.
- While an index is being updated, the rows involved can be **locked**, so other transactions wait.

In [ ]:
%%sql
REINDEX idx_species;

## 4. Your turn

Start from a clean table.

In [ ]:
reset()
print("Table reset.")

**TODO 1.** Add a new seedling: ID `S5`, species `Sea Holly`, tray `T-11`, stock `60`, price `6.00`. Name the columns.

In [ ]:
%%sql
-- TODO 1: INSERT INTO Seedling (...) VALUES (...)
-- Delete the next line when you have written your SQL.
SELECT 'write your SQL here' AS todo;

In [ ]:
check_db("TODO 1", "SELECT SeedlingID, SpeciesName, TrayCode, StockCount, Price FROM Seedling WHERE SeedlingID = 'S5'", [("S5", "Sea Holly", "T-11", 60, 6.0)])

**TODO 2.** The Silver Thyme (`S3`) is on special. Change its price to `3.00`. Make sure only **one** row changes.

In [ ]:
%%sql
-- TODO 2: UPDATE Seedling SET Price = ... WHERE ...
-- Delete the next line when you have written your SQL.
SELECT 'write your SQL here' AS todo;

In [ ]:
check_db("TODO 2", "SELECT SeedlingID, Price FROM Seedling ORDER BY SeedlingID", [("S1", 3.5), ("S2", 5.2), ("S3", 3.0), ("S4", 4.1), ("S5", 6.0)], ordered=True)

**TODO 3.** Remove every seedling that is out of stock (`StockCount` is 0).

In [ ]:
%%sql
-- TODO 3: DELETE FROM Seedling WHERE ...
-- Delete the next line when you have written your SQL.
SELECT 'write your SQL here' AS todo;

In [ ]:
check_db("TODO 3", "SELECT SeedlingID FROM Seedling ORDER BY SeedlingID", [("S1",), ("S2",), ("S3",), ("S5",)], ordered=True)

**TODO 4.** A delivery arrives: add **10** to the stock of every seedling in tray `T-04`. You can use the old value in the new one: `SET StockCount = StockCount + 10`.

In [ ]:
%%sql
-- TODO 4: UPDATE Seedling SET StockCount = StockCount + 10 WHERE TrayCode = ...
-- Delete the next line when you have written your SQL.
SELECT 'write your SQL here' AS todo;

In [ ]:
check_db("TODO 4", "SELECT SeedlingID, StockCount FROM Seedling WHERE TrayCode = 'T-04' ORDER BY SeedlingID", [("S1", 130), ("S3", 90)], ordered=True)

## 5. Check your understanding

`SEEDLING` has an index on `SpeciesName`. Which statement forces the database to do extra write work **inside the index**? **A** `UPDATE Seedling SET Price = 3.00 WHERE SeedlingID = 'S3'`, **B** `UPDATE Seedling SET SpeciesName = 'Garden Thyme' WHERE SeedlingID = 'S3'`, **C** `SELECT * FROM Seedling WHERE SpeciesName = 'Coral Bells'`, **D** both A and B, equally.

In [ ]:
choice = "?"   # type A, B, C or D
check_choice(choice, "B", "SpeciesName is the indexed column, so changing it means the old index entry is removed and a new one inserted in sorted position. Price is not indexed, so changing it touches the row only.",
             "The extra cost only appears when the value being changed is the one the index is built on. Which column is indexed?")

**Examiner note.** The command term *explain* needs a **mechanism**, not a claim. Weak: *"Updating an indexed column is slower because indexes make updates take longer."* That restates the question. Better: *"The row is written, and then the index must be maintained too: the old entry is removed and a new one inserted in sorted position. Repeated modification leaves the index fragmented, so it is rebuilt or reorganized to restore performance."*

## 6. Revision questions

The first (construct an INSERT) is TODO 1. Here are the other two.

**Q1. Explain what would happen if the WHERE clause were omitted from an UPDATE statement.** [3 marks]

Double-click this box and type your answer here, then press Shift+Enter.

> (your answer)

**Q2. Explain the performance implications of frequently updating an indexed column, and outline one way a database administrator can reduce them.** [5 marks]

Double-click this box and type your answer here, then press Shift+Enter.

> (your answer)